# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a recipe for loading, exploring, and processing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using the Croissant schema and is accessible at:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant

## 1. Data Loading

We will load the dataset metadata and inspect its top-level description.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic dataset metadata (name, description, version, citation, license)
meta = dataset.metadata
print(f"Name: {meta.name}\nDescription: {meta.description}\nVersion: {meta.version}")
print(f"Identifier: {meta.identifier}\nLicense: {meta.license}")

## 2. Data Overview

Inspect available record sets, fields, and their identifiers (`@id`). In Croissant datasets, each major data entity (record set, field, and column) is identified by its `@id`. 


In [ ]:
# List record sets (@id and name)
print("Record sets in dataset:")
record_sets = dataset.metadata.record_sets

for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '[no name]')}")
    # List associated fields (@id, name, dataType)
    print("  Fields:")
    for field in rs.get('field', []):
        field_id = field.get('@id', '')
        fname = field.get('name', '')
        ftype = field.get('dataType', '')
        print(f"    - {field_id}: {fname} ({ftype})")
    print()

## 3. Data Extraction

We will extract tabular data from each record set by referencing their `@id`s. This section also demonstrates how to refer to specific fields by their ids.


In [ ]:
# For this dataset, let's enumerate available record set ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

# Load data from each record set (most datasets have one main record set)
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set {rs_id}")
    else:
        print(f"No records found for {rs_id}")

# For demonstration, pick the first record set with data
main_rs_id = next(iter(dataframes))
print(f"\nFields in main record set ({main_rs_id}):\n", dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's process fields for further analysis. This includes filtering records, normalizing a numeric field, and grouping by key demographics (using field `@id`s explicitly as the column names).

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` below with actual ids (from the overview above) for meaningful analysis. The example below demonstrates usage with standard fields from the mental health survey (e.g., gad7_total, age, level_of_education).


In [ ]:
# Example: Let's pick two likely fields (by their @id, e.g., 'gad7_total', 'age', 'level_of_education')
# Adjust to match actual ids as shown from overview above.
numeric_field = 'gad7_total'   # Example field id (anxiety score)
group_field = 'level_of_education'  # E.g. educational level
rs_id = main_rs_id

df = dataframes[rs_id]

# Check presence
if numeric_field in df.columns:
    threshold = 7
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Records with {numeric_field} > {threshold}: {len(filtered_df)}")

    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

    print(filtered_df[[numeric_field, norm_col]].head())

    # Group by group_field if available
    if group_field in filtered_df.columns:
        grouped = (
            filtered_df.groupby(group_field)[numeric_field]
            .agg(['mean', 'std', 'count'])
            .reset_index()
        )
        print(f"\nGrouped summary by {group_field}:")
        print(grouped)
    else:
        print(f"Field '{group_field}' not available for grouping.")
else:
    print(f"Field '{numeric_field}' not found in record set '{rs_id}'.")

## 5. Visualization

Visualize the distribution of a core numeric field (e.g., GAD-7 scores or PHQ-9 scores) and how it relates to a demographic attribute (e.g., education).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the selected numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

# Boxplot by group field if present
if group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and interrogate the FAIR² Kilifi County mental health survey dataset using `mlcroissant`.

- **Metadata and structure**: The datset is well-documented, with record set and field `@id` metadata.
- **EDA**: We analyzed and visualized key assessment fields (e.g., GAD-7 scores). Filtering, normalization, and grouping showed how mental health indicators relate to demographic factors.
- **Next steps**: Further work could include modeling, longitudinal analyses, or combining across multiple assessment fields for a holistic view.

For more documentation on Croissant: https://mlcommons.org/croissant/
